# 정수 이론을 읽고 실제 계산 확인하기

아래 개념을 먼저 읽고 한 장 재현으로 확인합니다. 전체 모델 평가와 FPGA 검증은 서로 다른 단계입니다.

## 먼저 개념을 정리합니다

**정수화는 소수를 정수와 눈금으로 표현하는 일**입니다. 눈금(Scale)이 0.1이면 code 3은 실제 값 0.3입니다.

- 다른 눈금의 값은 단위를 맞춰 더합니다. cm와 m를 맞추는 것과 같습니다.
- 여러 곱을 더한 누산값은 클 수 있어 넓게 계산합니다.
- 출력 눈금으로 옮길 때 반올림하고, 범위를 넘으면 끝값에 붙입니다.
- RNE는 가까운 정수, 정확히 중간이면 짝수로 갑니다. 2.5→2, 3.5→4입니다.

실제 모델의 Weight·Bias·반올림 규칙을 고정한 기준이 E2-B입니다. 같은 입력과 상수로 계산하면 지정한 중간값과 출력이 같아야 합니다.

개념이 낯설면 [A6 정수 눈금](https://app.notion.com/p/407f0c4baec981e088c5ed706bc1a9b1)과 [D1 회로 숫자 규칙](https://app.notion.com/p/407f0c4baec981b8ae88d705e9af515b)을 먼저 읽습니다.

**이제 아래에서 작은 수를 계산하고, 실제 정수 모델 한 장으로 확인합니다.**

## 실행 전 짧은 예측

Main code 8·Scale 1/8과 Skip code 4·Scale 1/4의 합을 출력 Scale 1/8로 표현하면 12일까요, 16일까요? 실제 Stem 첫 출력의 곱의 합 411과 Bias 11879를 더하면 얼마일까요?
[정수 단위 그림](https://github.com/tsi03048/resnet20-accel/blob/main/docs/learning/images/integer_units.png)

In [ ]:
# 0. E1/E2-A 실습 폴더와 분리해 준비한다.
import os, sys, subprocess, json, importlib.util
from pathlib import Path
repo=Path('/content/resnet20-accel-e2b') if Path('/content').is_dir() else Path.cwd() / 'resnet20-accel-e2b'
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/tsi03048/resnet20-accel.git',str(repo)],check=True)
os.chdir(repo)
if not Path('quant/replay_e2b.py').exists():
    raise RuntimeError('이 폴더는 이전 버전입니다. 작업을 저장하고 새 런타임/폴더에서 다시 시작하세요.')
sys.path.insert(0,str(repo))
if importlib.util.find_spec('numpy') is None:
    subprocess.run([sys.executable,'-m','pip','install','numpy'],check=True)
subprocess.run(['git','rev-parse','HEAD'],check=True)


In [ ]:
# 1. 파일 검사를 통과한 상수와 입력으로 실제 정수 계산
from quant.replay_e2b import replay
from datetime import datetime, timezone
result=replay(backend='numpy')
result['executed_at_utc']=datetime.now(timezone.utc).isoformat()
Path('results/e2b_student_replay.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps(result,indent=2))


정상 결과: PASS / 61경계 / 552,084원소 / prediction 3 / cat.
이것은 소프트웨어 Golden 재현이며 RTL·보드 통과가 아니다.


In [ ]:
# 2. 실제 Stem 첫 원소를 따라가기
import numpy as np
from quant.integer_math import rne_div
with np.load('reference/e2b_v1/constants.npz',allow_pickle=False) as z:
    bias=int(z['stem.bias'][0]); n=int(z['stem.multiplier'][0]); shift=int(z['stem.shift'][0])
with np.load('reference/e2b_v1/sample0/tensors.npz',allow_pickle=False) as z:
    acc=int(z['stem.acc'][0,0,0,0]); expected=int(z['stem'][0,0,0,0])
code=min(127,max(0,rne_div(acc*n,1<<shift)))
print('곱의 합:',acc-bias,'Bias:',bias,'acc:',acc,'N:',n,'shift:',shift)
print('직접 계산:',code,'Golden:',expected)
if code!=expected: raise RuntimeError('첫 Stem code 불일치')


## 관찰 직후 확인

1. 위 Stem 출력 20은 어떤 누산값과 계수에서 나왔나요?
2. Shift만 적용하는 계산과 RNE는 왜 다를 수 있나요?
3. 한 장의 61경계 일치가 10,000장 정확도나 FPGA timing을 뜻하나요?

In [ ]:
from quant.integer_math import rne_div
print('동점 반올림:', [rne_div(v,2) for v in (5,7,-5,-7)])
print('기대값: [2,4,-2,-4]')

## 선택: 상수부터 재생성하고 Test 10,000장 평가

아래 `RUN_FULL`을 True로 바꾸면 체크포인트·CIFAR 데이터를 받고 전체 평가한다.
배포한 E2-A 출처를 고정해 읽으며 새로 선택하지 않는다. CPU에서 시간이 걸린다.
빠른 16장 점검은 전체 정확도 결과로 제출하지 않는다.


In [ ]:
RUN_FULL=False
if RUN_FULL:
    if any(importlib.util.find_spec(p) is None for p in ('torch','torchvision','datasets')):
        subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
    subprocess.run([sys.executable,'-m','quant.run_e2b','--limit','10000','--audit','8','--batch-size','32'],check=True)
    print(Path('results/e2b_integer.json').read_text(encoding='utf-8'))
else:
    print('한 장 재현만 실행했습니다. results/e2b_integer.json은 저장소에 배포된 기준 실행 기록입니다.')


## 제출

실행자·날짜·패키지 hash·실행한 범위·첫 Stem 계산·확인 질문 답안을 함께 제출한다.
ZIP에는 저장소의 기존 결과도 들어 있다. 본인 실행은 `e2b_student_replay.json`,
전체 실행은 `e2b_run_manifest.json`의 시간·인자로 구분한다.


In [ ]:
# 3. 결과·상수·Golden 보관
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile('e2b_outputs.zip','w',ZIP_DEFLATED) as z:
    for folder in ('reference/e2b_v1','results','artifacts/e2b_v1','golden/e2b_sample0'):
        for path in Path(folder).rglob('*'):
            if path.is_file(): z.write(path)
try:
    from google.colab import files
    files.download('e2b_outputs.zip')
except ImportError:
    print(Path('e2b_outputs.zip').resolve())
